# 03 – Granger Causality & VAR

Test whether macro variables Granger-cause credit spreads, fit a Vector Autoregression, and compute Impulse Response Functions and Variance Decompositions.

**Covers:** Granger causality tests, VAR model fitting, IRF charts, FEVD, Johansen cointegration.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from config.settings import DEFAULT_START_DATE, DEFAULT_END_DATE, FRED_API_KEY, DATA_DIR
from src.data.fetcher import fetch_all_data

df = fetch_all_data(
    start_date=DEFAULT_START_DATE,
    end_date=DEFAULT_END_DATE,
    api_key=FRED_API_KEY,
    cache_dir=DATA_DIR,
)
print(f'Data loaded: {df.shape}')

In [ ]:
from src.models.statistical import run_granger_causality, summarize_granger

pairs = [
    ('hy_spread', 'vix'),
    ('hy_spread', 't10y2y'),
    ('hy_spread', 'fed_funds'),
]

df_clean = df.dropna()
for caused, causing in pairs:
    if caused in df_clean.columns and causing in df_clean.columns:
        pvals = run_granger_causality(df_clean, caused=caused, causing=causing, maxlag=10)
        summarize_granger(pvals, caused=caused, causing=causing)
        print()

In [ ]:
from src.models.statistical import fit_var_model, compute_irf

var_cols = [c for c in ['hy_spread', 'vix', 't10y2y'] if c in df.columns]
print('VAR columns:', var_cols)

var_result = fit_var_model(df, columns=var_cols, maxlags=10)
print(var_result.summary())

In [ ]:
# Impulse Response Functions
irf = compute_irf(var_result, periods=20)

from src.visualization.plots import plot_impulse_response
fig = plot_impulse_response(irf)
fig.show()

In [ ]:
# Forecast Error Variance Decomposition
from src.models.statistical import compute_variance_decomposition

fevd = compute_variance_decomposition(var_result, periods=20)
print('FEVD summary:')
fevd.summary()

In [ ]:
# Johansen cointegration test
from src.models.statistical import run_johansen_cointegration

spread_cols = [c for c in ['hy_spread', 'ig_spread', 'bbb_spread'] if c in df.columns]
if len(spread_cols) >= 2:
    johansen = run_johansen_cointegration(df, columns=spread_cols)
    print('Trace statistics:', johansen.lr1)
    print('Critical values (5%):', johansen.cvt[:, 1])